# Final DS

## Objective

Develop a final training dataset based on the variables selected in `WCExperience.ipynb` and `GameConditions.ipynb`.

## Build final dataset

Final model variables included:

- `goals_for`
- `ELO_diff`
- `opp_prior_world_cups_coached`
- `opp_players_with_multiple_prior_wcs`
- `avg_age^2`
- `opp_avg_age`
- `opp_avg_age^2`
- `SouthAfrica`
- `distance_from_host_km`


In [1]:
library(tidyverse)
library(here)

Warning message:
"package 'ggplot2' was built under R version 4.4.3"
Warning message:
"package 'purrr' was built under R version 4.4.3"
-- Attaching core tidyverse packages ------------------------ tidyverse 2.0.0 --
v dplyr     1.1.4     v readr     2.1.5
v forcats   1.0.0     v stringr   1.6.0
v ggplot2   4.0.1     v tibble    3.2.1
v lubridate 1.9.4     v tidyr     1.3.1
v purrr     1.2.1     
-- Conflicts ------------------------------------------ tidyverse_conflicts() --
x dplyr::filter() masks stats::filter()
x dplyr::lag()    masks stats::lag()
i Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
here() starts at /Users/eialnisman/Desktop/WC2026Forecast



## Data

I start from `FullTeamGames` for team-match rows, then join tournament-specific Elo ratings, distance from host country, opponent manager World Cup history, and player experience/age for both teams.


In [2]:
Games <- readRDS(here("1.DataCleaning-R", "Data", "RDS", "FullTeamGames.rds"))
ELO <- readRDS(here("1.DataCleaning-R", "Data", "RDS", "ELOSScores.rds"))
DistanceFromHome <- readRDS(here("1.DataCleaning-R", "Data", "RDS", "DistanceFromHome.rds"))
ManagerHistory <- readRDS(here("1.DataCleaning-R", "Data", "RDS", "ManagerWC_history.rds"))
PlayerExperience <- readRDS(here("1.DataCleaning-R", "Data", "RDS", "PlayerExperience.rds"))


Elo scores are stored wide by tournament, so I reshape them into one row per country and World Cup before joining team and opponent ratings.

In [3]:
ELOByTournament <- ELO %>%
  pivot_longer(
    cols = starts_with("WC_"),
    names_to = "tournament_year",
    values_to = "elo"
  ) %>%
  mutate(tournament_id = str_replace(tournament_year, "_", "-")) %>%
  select(team, tournament_id, elo)

head(ELOByTournament)

team,tournament_id,elo
<chr>,<chr>,<int>
South Africa,WC-2026,1550
South Africa,WC-2022,1548
South Africa,WC-2018,1541
South Africa,WC-2014,1573
South Africa,WC-2010,1519
Mexico,WC-2026,1835


Now I build the backtest dataset for the completed World Cups: 2010, 2014, 2018, and 2022.


In [4]:
completed_tournaments <- c("WC-2010", "WC-2014", "WC-2018", "WC-2022")

OpponentManagerHistory <- ManagerHistory %>%
  select(
    tournament_id,
    opponent_id = team_id,
    opp_prior_world_cups_coached = prior_world_cups_coached
  )

TeamPlayerExperience <- PlayerExperience %>%
  select(
    tournament_id,
    team_id,
    avg_age
  )

OpponentPlayerExperience <- PlayerExperience %>%
  select(
    tournament_id,
    opponent_id = team_id,
    opp_avg_age = avg_age,
    opp_players_with_multiple_prior_wcs = players_with_multiple_prior_wcs
  )

BacktestDataset <- Games %>%
  filter(tournament_id %in% completed_tournaments) %>%
  left_join(
    ELOByTournament %>% rename(team_name = team, team_ELO = elo),
    by = c("team_name", "tournament_id")
  ) %>%
  left_join(
    ELOByTournament %>% rename(opponent_name = team, opponent_ELO = elo),
    by = c("opponent_name", "tournament_id")
  ) %>%
  left_join(
    DistanceFromHome %>%
      select(tournament_id, team_id, distance_from_host_km),
    by = c("tournament_id", "team_id")
  ) %>%
  left_join(
    OpponentManagerHistory,
    by = c("tournament_id", "opponent_id")
  ) %>%
  left_join(
    TeamPlayerExperience,
    by = c("tournament_id", "team_id")
  ) %>%
  left_join(
    OpponentPlayerExperience,
    by = c("tournament_id", "opponent_id")
  ) %>%
  mutate(
    team = team_name,
    ELO_diff = team_ELO - opponent_ELO,
    SouthAfrica = if_else(tournament_id == "WC-2010", 1, 0),
    avg_age_squared = avg_age^2,
    opp_avg_age_squared = opp_avg_age^2
  ) %>%
  select(
    tournament_id,
    match_id,
    team_id,
    team,
    opponent_id,
    opponent_name,
    goals_for,
    goals_against,
    team_ELO,
    opponent_ELO,
    ELO_diff,
    opp_prior_world_cups_coached,
    opp_players_with_multiple_prior_wcs,
    avg_age,
    avg_age_squared,
    opp_avg_age,
    opp_avg_age_squared,
    SouthAfrica,
    distance_from_host_km
  ) %>%
    filter(match_id!="M-2014-61")
    
  

Backtest_WC_2010 <- BacktestDataset %>% filter(tournament_id == "WC-2010")
Backtest_WC_2014 <- BacktestDataset %>% filter(tournament_id == "WC-2014")
Backtest_WC_2018 <- BacktestDataset %>% filter(tournament_id == "WC-2018")
Backtest_WC_2022 <- BacktestDataset %>% filter(tournament_id == "WC-2022")

head(BacktestDataset)


tournament_id,match_id,team_id,team,opponent_id,opponent_name,goals_for,goals_against,team_ELO,opponent_ELO,ELO_diff,opp_prior_world_cups_coached,opp_players_with_multiple_prior_wcs,avg_age,avg_age_squared,opp_avg_age,opp_avg_age_squared,SouthAfrica,distance_from_host_km
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<int>,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
WC-2010,M-2010-06,T-01,Algeria,T-69,Slovenia,0,1,1580,1673,-93,0,0,26.69744,712.7532,27.19357,739.4901,1,7644.403
WC-2010,M-2010-23,T-01,Algeria,T-28,England,0,0,1580,1985,-405,0,2,26.69744,712.7532,28.93389,837.1699,1,7644.403
WC-2010,M-2010-38,T-01,Algeria,T-83,United States,0,1,1580,1802,-222,0,2,26.69744,712.7532,27.27373,743.8566,1,7644.403
WC-2010,M-2010-04,T-03,Argentina,T-50,Nigeria,1,0,1897,1747,150,2,1,27.61965,762.8453,26.25027,689.0766,1,8219.664
WC-2010,M-2010-18,T-03,Argentina,T-71,South Korea,4,1,1897,1749,148,0,5,27.61965,762.8453,27.58320,760.8332,1,8219.664
WC-2010,M-2010-35,T-03,Argentina,T-33,Greece,2,0,1897,1782,115,0,0,27.61965,762.8453,28.08767,788.9173,1,8219.664


Lets check dimensions, missing values, and tournament coverage before saving.

In [5]:
BacktestDataset %>%
  count(tournament_id)

BacktestDataset %>%
  summarise(
    rows = n(),
    missing_team_ELO = sum(is.na(team_ELO)),
    missing_opponent_ELO = sum(is.na(opponent_ELO)),
    missing_ELO_diff = sum(is.na(ELO_diff)),
    missing_opp_prior_world_cups_coached = sum(is.na(opp_prior_world_cups_coached)),
    missing_opp_players_with_multiple_prior_wcs = sum(is.na(opp_players_with_multiple_prior_wcs)),
    missing_avg_age = sum(is.na(avg_age)),
    missing_opp_avg_age = sum(is.na(opp_avg_age)),
    missing_SouthAfrica = sum(is.na(SouthAfrica)),
    missing_distance_from_host_km = sum(is.na(distance_from_host_km))
  )

list(
  Backtest_WC_2010 = dim(Backtest_WC_2010),
  Backtest_WC_2014 = dim(Backtest_WC_2014),
  Backtest_WC_2018 = dim(Backtest_WC_2018),
  Backtest_WC_2022 = dim(Backtest_WC_2022)
)


tournament_id,n
<chr>,<int>
WC-2010,128
WC-2014,126
WC-2018,128
WC-2022,128


rows,missing_team_ELO,missing_opponent_ELO,missing_ELO_diff,missing_opp_prior_world_cups_coached,missing_opp_players_with_multiple_prior_wcs,missing_avg_age,missing_opp_avg_age,missing_SouthAfrica,missing_distance_from_host_km
<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
510,0,0,0,0,0,0,0,0,0


$Backtest_WC_2010
[1] 128  19

$Backtest_WC_2014
[1] 126  19

$Backtest_WC_2018
[1] 128  19

$Backtest_WC_2022
[1] 128  19

Looks good.

In [6]:
saveRDS(BacktestDataset, here("1.DataCleaning-R", "Data", "RDS", "FinalTrainingDS.rds"))